# Notebook 01 — Cohort, splits and compact cache

**Purpose.** Build patient/recording connected components, freeze the 60/10/5/5/20 allocation, audit leakage, convert the paired spectral cache (float16 + packed validity) and fit the train-only normalizer. **Inputs:** validated source manifest. **Outputs:** `private/manifests/split_*.json`, `private/cache/<preprocess_hash>/`. **Partitions:** label summaries only (no model predictions). **GPU:** off.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
REPO = Path.cwd().resolve() if (Path.cwd() / "src" / "cape_eeg").exists() else Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("CAPE_ROOT", str(REPO.parent)); os.environ.setdefault("HMS_DATA_ROOT", os.environ["CAPE_ROOT"])
os.environ["PYTHONWARNINGS"] = "ignore"
from cape_eeg.paths import resolve_workspace, redact
from cape_eeg.status import read_json, Ledger
ws = resolve_workspace()
def run(cmd, **kw):
    """Run a repository script as a bounded subprocess; prints filtered output (no secrets, no identifiers)."""
    p = subprocess.run([sys.executable, str(REPO / "scripts" / cmd[0]), *cmd[1:]], capture_output=True, text=True, env=os.environ, **kw)
    for line in (p.stdout + p.stderr).splitlines():
        if line.strip() and not any(w in line for w in ("Warning", "warn", "Found GPU", "Minimum and", "(8.0)")):
            print(line)
    if p.returncode != 0:
        raise RuntimeError(f"{cmd[0]} exited with {p.returncode}")
print("repo:", redact(REPO, ws)); print("workspace root:", redact(ws.root, ws)); print("data root:", redact(ws.data, ws)); print("private:", redact(ws.private, ws))


## Connected components and deterministic allocation (seed 20260907)

In [ ]:
run(['make_splits.py'])
la = read_json(ws.manifests / 'leakage_audit.json'); print('leakage audit:', la['status'], 'forbidden overlaps:', la['forbidden_overlap'])
print('patient 5x5 intersection matrix (rows/cols train,tune,cal-T,cal-P,test):'); for r in la['patient_id']['matrix']: print(r)

## One-shard smoke then full bounded conversion
The converter writes `.partial` shards and renames them atomically only after both passes complete for every row. Re-running reuses a finished cache with the same preprocess hash.

In [ ]:
from cape_eeg.contracts import preprocess_signature, stable_hash
phash = stable_hash(preprocess_signature()); cache_dir = ws.cache / phash
if (cache_dir / 'manifest.json').exists():
    print('cache already complete for preprocess hash', phash)
else:
    run(['build_cache.py', '--smoke', '64', '--workers', '8'])
    run(['build_cache.py', '--workers', '16'])
man = read_json(cache_dir / 'manifest.json')
print({k: man[k] for k in ['rows','active_cache_bytes','alt_uniform_bytes','within_cap','rows_local_all_invalid','rows_context_all_invalid','rows_both_invalid','cache_hash']})
print('conversion resources:', read_json(cache_dir / 'conversion_resources.json'))

## Float16 round-trip and validity summary

In [ ]:
import numpy as np
from cape_eeg.data.cache import CacheReader
from cape_eeg.data.spectral import unpack_mask
r = CacheReader(cache_dir); q = r.quality
print(q[['local_valid_fraction','context_valid_fraction','lead_valid_fraction','eeg_observed_fraction']].describe().loc[['mean','min','50%']].round(4))
rows = np.sort(np.random.default_rng(0).choice(r.n, 256, replace=False)); b = r.rows(rows)
print('float16 storage; values range local', float(b['local'].min()), float(b['local'].max()), 'context', float(b['context'].min()), float(b['context'].max()))
print('cache bytes vs cap: %.3f GB of 6.000 GB' % (man['active_cache_bytes'] / 1e9))

## Train-only normalizer
Fitted on a bounded deterministic sample of training rows; refit on train+tune only for the final phase (handled by `scripts/train.py`).

In [ ]:
from cape_eeg.data.dataset import select_rows
from cape_eeg.data.normalization import fit_normalizer, save_normalizer
split_hash = read_json(ws.manifests / 'split_summary.json')['split_hash']
for enc, alt in [('foveated', False), ('uniform', True)]:
    p = ws.normalization / split_hash / f'{phash}_train_{enc}.json'
    if not p.exists():
        save_normalizer(fit_normalizer(CacheReader(cache_dir, alt_uniform=alt), select_rows(r.index, ['train'])), p)
    n = read_json(p); print(enc, {v: [(round(s['median'], 2), round(s['scale'], 2)) for s in st] for v, st in n['views'].items()})